<a href="https://colab.research.google.com/github/ritashreemukherjee123/Project-1-Applied-Search-Intelligence-Google-Search-Ranking-Discoverability.-/blob/main/work/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

The task is to prioritize content pages for review, so the model should produce a ranking score rather than only a hard yes/no prediction.

I will start with Logistic Regression because the target is binary and the model provides a simple, interpretable learned benchmark. I will then compare it with Random Forest to test whether nonlinear feature interactions improve the ranking.

The model will use predicted probability as the priority score. I will evaluate the ranking using the same split and primary metric as the Week-4 baseline.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
df = pd.read_parquet("/content/drive/MyDrive/colab_outputs/features.parquet")
print(df.head())

  as_of_date           client_hash_id           content_hash_id  \
0 2026-06-01  client_06d356715a8ff3b6  content_045a69625ade61a8   
1 2026-06-15  client_06d356715a8ff3b6  content_045a69625ade61a8   
2 2026-06-29  client_06d356715a8ff3b6  content_045a69625ade61a8   
3 2026-06-01  client_06d356715a8ff3b6  content_09063e5eace59644   
4 2026-06-15  client_06d356715a8ff3b6  content_09063e5eace59644   

   impressions_90d  impressions_last30  impressions_prev30  clicks_last30  \
0            115.0               115.0                 NaN            2.0   
1           2868.0              2868.0                 NaN            3.0   
2           4785.0              4785.0                 NaN            5.0   
3            605.0               605.0                 NaN           18.0   
4           9790.0              9790.0                 NaN          254.0   

   clicks_prev30  avg_position_last30  ga4_sessions_last30  ...  \
0            NaN            10.721739                  2.0  ...   


In [5]:
print(df.groupby("as_of_date").agg(rows=("is_declining","size"),
      base_rate=("is_declining","mean"),
      active_days=("active_days_90d","mean")))

             rows  base_rate  active_days
as_of_date                               
2026-06-01  17992   0.064973     1.000000
2026-06-15  62550   0.585148    14.940943
2026-06-29  76236   0.989572    28.594890


In [7]:
from sklearn.metrics import roc_auc_score

# 1) Does a single date-proxy feature predict the label by itself?
print(roc_auc_score(df["is_declining"], df["active_days_90d"]))



0.8806223221558909


In [8]:
from sklearn.metrics import roc_auc_score
for d, g in df.groupby("as_of_date"):
    if g["is_declining"].nunique() == 2:
        print(d.date(), len(g), "active_days AUC:",
              round(roc_auc_score(g["is_declining"], g["active_days_90d"]), 3))

2026-06-01 17992 active_days AUC: 0.5
2026-06-15 62550 active_days AUC: 0.506
2026-06-29 76236 active_days AUC: 0.538


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

We have several features for the task here. `as_of_date` column is crucial for time-aware split.



In [9]:
## Using 06-15 only, as a provisional within-date check, with the client-grouped split

from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

d = df[df["as_of_date"] == "2026-06-15"]
DROP = ["as_of_date","client_hash_id","content_hash_id","is_declining","clicks_future30"]
X, y, g = d.drop(columns=DROP).astype(float), d["is_declining"], d["client_hash_id"]

aucs = []
for tr, te in GroupKFold(5).split(X, y, g):
    m = RandomForestClassifier(300, min_samples_leaf=20, n_jobs=-1, random_state=0)
    m.fit(X.iloc[tr], y.iloc[tr])
    aucs.append(roc_auc_score(y.iloc[te], m.predict_proba(X.iloc[te])[:, 1]))
print("base rate", round(y.mean(), 3), "AUC", [round(a, 3) for a in aucs])

base rate 0.585 AUC [np.float64(0.674), np.float64(0.65), np.float64(0.688), np.float64(0.694), np.float64(0.66)]


In [12]:
import duckdb

# Initialize DuckDB connection
con = duckdb.connect()

# Define TABLES mapping to the 'df' DataFrame, as 'fact_daily_sample' and 'fact_daily'
# are not directly available and 'df' is the current focus.
TABLES = {
    "fact_daily_sample": "df",
    "fact_daily": "df"
}

for t in ["fact_daily_sample", "fact_daily"]:
    # The original query used 'report_date', but 'df' has 'as_of_date'.
    # We'll use 'as_of_date' to get the date range from the loaded features DataFrame.
    print(t, con.sql(f"""SELECT MIN(as_of_date) mn, MAX(as_of_date) mx,
          COUNT(DISTINCT as_of_date) AS "days" FROM {TABLES[t]}""").df().to_dict("records"))

fact_daily_sample [{'mn': Timestamp('2026-06-01 00:00:00'), 'mx': Timestamp('2026-06-29 00:00:00'), 'days': 3}]
fact_daily [{'mn': Timestamp('2026-06-01 00:00:00'), 'mx': Timestamp('2026-06-29 00:00:00'), 'days': 3}]


In [14]:
for t in ["fact_daily_sample", "fact_daily"]:
    print(t, con.sql(f"""
        SELECT as_of_date, COUNT(*) AS "rows", COUNT(DISTINCT content_hash_id) AS "pages"
        FROM {TABLES[t]} GROUP BY 1 ORDER BY 1""").df().head(35))

fact_daily_sample   as_of_date   rows  pages
0 2026-06-01  17992  17992
1 2026-06-15  62550  62550
2 2026-06-29  76236  76151
fact_daily   as_of_date   rows  pages
0 2026-06-01  17992  17992
1 2026-06-15  62550  62550
2 2026-06-29  76236  76151


In [15]:
from sklearn.metrics import roc_auc_score
d = df[df["as_of_date"] == "2026-06-15"]
for c in ["clicks_last30", "impressions_last30", "avg_position_last30",
          "content_age_days_at_as_of"]:
    print(c, round(roc_auc_score(d["is_declining"], d[c]), 3))

for lo in [1, 5, 10, 30]:
    s = d[d["clicks_last30"] >= lo]
    print(f"clicks_last30>={lo}: n={len(s)}, base_rate={s['is_declining'].mean():.3f}")

clicks_last30 0.482
impressions_last30 0.387
avg_position_last30 0.575
content_age_days_at_as_of 0.532
clicks_last30>=1: n=62550, base_rate=0.585
clicks_last30>=5: n=20611, base_rate=0.555
clicks_last30>=10: n=10436, base_rate=0.529
clicks_last30>=30: n=2647, base_rate=0.482


In [18]:
print(TABLES["fact_daily_sample"])   # should be a read_parquet('hf://...') string
daily = con.sql(f"""
  SELECT as_of_date, COUNT(*) AS n_rows, SUM(clicks_last30) AS clicks,
         SUM(impressions_last30) AS impr
  FROM {TABLES['fact_daily_sample']} GROUP BY 1 ORDER BY 1""").df()
print(daily)

df
  as_of_date  n_rows     clicks         impr
0 2026-06-01   17992    36784.0    5142732.0
1 2026-06-15   62550   634707.0  104846812.0
2 2026-06-29   76236  1084432.0  193851354.0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [21]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

max_date = pd.Timestamp(daily["as_of_date"].max())
as_of = pd.Timestamp("2026-06-15")
past_days = 15                       # 06-01..06-15 inclusive
future_days = (max_date - as_of).days

d = df[(df["as_of_date"] == as_of) & (df["clicks_last30"] >= 10)].copy()
d["y_rate"] = ((d["clicks_future30"] / future_days)
               < 0.7 * (d["clicks_last30"] / past_days)).astype(int)

DROP = ["as_of_date","client_hash_id","content_hash_id","is_declining",
        "clicks_future30","y_rate"]
X, y, g = d.drop(columns=DROP).astype(float), d["y_rate"], d["client_hash_id"]

aucs = []
for tr, te in GroupKFold(5).split(X, y, g):
    m = RandomForestClassifier(300, min_samples_leaf=20, n_jobs=-1, random_state=0)
    m.fit(X.iloc[tr], y.iloc[tr])
    aucs.append(roc_auc_score(y.iloc[te], m.predict_proba(X.iloc[te])[:, 1]))
print(len(d), "base rate", round(y.mean(), 3), [round(a, 3) for a in aucs])

10436 base rate 0.49 [np.float64(0.552), np.float64(0.559), np.float64(0.655), np.float64(0.593), np.float64(0.62)]


In [23]:
## Recompute the W04 score on d, using the same thresholds, so the baseline and the models are compared on identical rows and the same y

VISIBLE_IMPRESSIONS = 500
DECLINE_RATIO_THRESHOLD = 0.80
STALE_DAYS = 180

d = d.copy()
d["impression_ratio"] = d["impressions_last30"] / d["impressions_prev30"]
d["baseline_score"] = (
    (d["impressions_last30"] < VISIBLE_IMPRESSIONS).astype(int)
    + (d["impression_ratio"] < DECLINE_RATIO_THRESHOLD).astype(int)
    + (d["content_age_days_at_as_of"] >= STALE_DAYS).astype(int)
)

print(d["baseline_score"].value_counts().sort_index())
print("baseline AUC:", round(roc_auc_score(d["y_rate"], d["baseline_score"]), 3))

# precision@k: top 100 by baseline, ties broken by lowest impressions (as in W04)
top = d.sort_values(["baseline_score", "impressions_last30"], ascending=[False, True]).head(100)
print("baseline precision@100:", round(top["y_rate"].mean(), 3), "vs base rate", round(d["y_rate"].mean(), 3))

baseline_score
0    5866
1    4521
2      49
Name: count, dtype: int64
baseline AUC: 0.521
baseline precision@100: 0.64 vs base rate 0.49


In [25]:
## Scoring the models with the same precision@100, on the same rows, and report the rule's AUC without the tie-break

# random forest / logreg out-of-fold predictions on the same folds
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X = X.fillna(0) # Fill NaN values with 0 before model training

def oof(model):
    p = np.zeros(len(d))
    for tr, te in GroupKFold(5).split(X, y, g):
        model.fit(X.iloc[tr], y.iloc[tr])
        p[te] = model.predict_proba(X.iloc[te])[:, 1]
    return p

rf_p = oof(RandomForestClassifier(300, min_samples_leaf=20, n_jobs=-1, random_state=0))
lr_p = oof(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)))

def p_at_k(score, k=100):
    return y.iloc[np.argsort(-np.asarray(score))[:k]].mean()

for name, s in [("baseline (tie-broken)", -d["impressions_last30"] + 1e6 * d["baseline_score"]),
                ("rf", rf_p), ("logreg", lr_p),
                ("-impressions_last30", -d["impressions_last30"])]:
    print(name, "AUC", round(roc_auc_score(y, s), 3), "P@100", round(p_at_k(s), 3))

baseline (tie-broken) AUC 0.57 P@100 0.64
rf AUC 0.52 P@100 0.58
logreg AUC 0.518 P@100 0.66
-impressions_last30 AUC 0.595 P@100 0.63


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


#What it leans on?
##Use permutation importance on held-out folds, scored by AUC. Impurity importance from a
## random forest favors high-cardinality, noisy columns, so avoid it here.
from sklearn.inspection import permutation_importance
imps = []
for tr, te in GroupKFold(5).split(X, y, g):
    m = RandomForestClassifier(300, min_samples_leaf=20, n_jobs=-1, random_state=0).fit(X.iloc[tr], y.iloc[tr])
    r = permutation_importance(m, X.iloc[te], y.iloc[te], scoring="roc_auc",
                               n_repeats=5, random_state=0, n_jobs=-1)
    imps.append(r.importances_mean)
imp = pd.Series(np.mean(imps, axis=0), index=X.columns).sort_values(ascending=False)
print(imp.head(8).round(4))


avg_position_last30          0.0339
impressions_last30           0.0179
impressions_90d              0.0171
ga4_sessions_last30          0.0061
ctr_last30                   0.0052
url_char_count               0.0044
keyword_token_count          0.0031
main_intent_transactional    0.0004
dtype: float64


In [27]:
##Where is it wrong?
## Slice the out-of-fold predictions by the things that plausibly drive noise.

d["p"] = rf_p
def slice_report(col, bins):
    s = pd.cut(d[col], bins)
    out = d.groupby(s, observed=True).apply(lambda t: pd.Series({
        "n": len(t), "actual_rate": t["y_rate"].mean(), "mean_pred": t["p"].mean(),
        "auc": roc_auc_score(t["y_rate"], t["p"]) if t["y_rate"].nunique() == 2 else np.nan}))
    print(col); print(out.round(3), "\n")

slice_report("clicks_last30", [9, 20, 50, 200, np.inf])
slice_report("avg_position_last30", [0, 3, 7, 12, 20, 100])
slice_report("content_age_days_at_as_of", [-2, 90, 180, 365, 5000])

clicks_last30
                    n  actual_rate  mean_pred    auc
clicks_last30                                       
(9.0, 20.0]    6148.0        0.524      0.505  0.512
(20.0, 50.0]   3187.0        0.468      0.456  0.494
(50.0, 200.0]  1006.0        0.365      0.395  0.387
(200.0, inf]     95.0        0.337      0.395  0.312 

avg_position_last30
                          n  actual_rate  mean_pred    auc
avg_position_last30                                       
(0, 3]                347.0        0.349      0.432  0.586
(3, 7]               5988.0        0.421      0.451  0.510
(7, 12]              2703.0        0.549      0.518  0.439
(12, 20]              904.0        0.701      0.553  0.424
(20, 100]             494.0        0.715      0.489  0.443 

content_age_days_at_as_of
                                n  actual_rate  mean_pred    auc
content_age_days_at_as_of                                       
(-2, 90]                   3090.0        0.419      0.492  0.552
(90, 180] 

In [28]:
# 1) Does the simple rank rule beat the forest at the top?
hi = d[d["avg_position_last30"] > 12]
print("position>12:", len(hi), round(hi["y_rate"].mean(), 3))
for k in (20, 100, 500):
    print(k, "position desc:", round(d.nlargest(k, "avg_position_last30")["y_rate"].mean(), 3),
             "rf:", round(d.nlargest(k, "p")["y_rate"].mean(), 3))

# 2) Does the rank effect hold across clients, or is it one big client?
print(hi.groupby("client_hash_id")["y_rate"].agg(["size", "mean"]).sort_values("size", ascending=False).head(8).round(3))

# 3) Mean-reversion test: raw clicks inside the high-click buckets
for lo, hi_ in [(50, 200), (200, 1e9)]:
    s = d[(d["clicks_last30"] > lo) & (d["clicks_last30"] <= hi_)]
    print(lo, round(roc_auc_score(s["y_rate"], s["clicks_last30"]), 3))

position>12: 1398 0.706
20 position desc: 0.65 rf: 0.6
100 position desc: 0.73 rf: 0.58
500 position desc: 0.714 rf: 0.526
                         size   mean
client_hash_id                      
client_23a62021009f63c4   866  0.771
client_e547b89c05043229   135  0.267
client_1a730cb2640a1abf   108  0.991
client_8ddc46da5414ffd8    86  0.547
client_20259bd6705d81d4    83  0.627
client_e5c2aa26a8598242    31  0.806
client_06d356715a8ff3b6    15  0.333
client_fef1a8f436438636    13  0.692
50 0.494
200 0.509


What the three outputs show

1. Without the two big clients, the effect shrinks a lot. Pages ranked worse than position 12 drop from 0.706 to 0.50, against a segment base rate of 0.435. That is a lift of about 6 points on 424 pages. It is real-looking but small, and since pages cluster within clients, it would be suggestive.

2. The top-k lists are mostly one client. client_23a6… supplies 82 of the top 100 and 386 of the top 500. So "position, worst first" is largely "pick client_23a6's pages".

3. That client's pages decline regardless of position. Its overall base rate is 0.761 (1,649 pages), and its poorly ranked pages decline at 0.771. There is essentially no within-client lift, and its position AUC is 0.525. Over half its pages (866 of 1,649) are poorly ranked, against about 13% overall. The pooled rule looks good because it sorts pages by a client that declines a lot, not because position predicts decline.

4. Within clients, position has a weak but consistent signal. Across the 15 clients with at least 100 pages, position AUC ranges from 0.50 to 0.60 (median about 0.59), and 13 of 15 are above 0.5. The one outlier is client_1a73… at 0.818.

In [29]:
big = ["client_23a62021009f63c4", "client_1a730cb2640a1abf"]

# 1) Position rule with and without the two big clients
for label, s in [("all", d), ("excl big 2", d[~d.client_hash_id.isin(big)])]:
    hi = s[s.avg_position_last30 > 12]
    print(label, len(hi), round(hi.y_rate.mean(), 3), "| segment base rate", round(s.y_rate.mean(), 3))

# 2) Who fills the top-k lists?
for k in (100, 500):
    print(k, d.nlargest(k, "avg_position_last30").client_hash_id.value_counts().head(4).to_dict())

# 3) Per-client AUC of position (is it >0.5 across clients, or just a few?)
rows = []
for c, t in d.groupby("client_hash_id"):
    if len(t) >= 100 and t.y_rate.nunique() == 2:
        rows.append((c[:14], len(t), round(t.y_rate.mean(), 3),
                     round(roc_auc_score(t.y_rate, t.avg_position_last30), 3)))
print(pd.DataFrame(rows, columns=["client", "n", "base", "pos_auc"]))

all 1398 0.706 | segment base rate 0.49
excl big 2 424 0.5 | segment base rate 0.435
100 {'client_23a62021009f63c4': 82, 'client_20259bd6705d81d4': 8, 'client_f623b01661d4bfe4': 3, 'client_8ddc46da5414ffd8': 3}
500 {'client_23a62021009f63c4': 386, 'client_e547b89c05043229': 32, 'client_20259bd6705d81d4': 31, 'client_8ddc46da5414ffd8': 25}
            client     n   base  pos_auc
0   client_06d3567   218  0.165    0.589
1   client_08a6a72   202  0.490    0.497
2   client_0fa64a1   170  0.435    0.498
3   client_1a730cb   428  0.514    0.818
4   client_20259bd   459  0.501    0.601
5   client_2094c6e   122  0.705    0.560
6   client_23a6202  1649  0.761    0.525
7   client_62f4a7e   937  0.602    0.592
8   client_73cda7b  2097  0.458    0.597
9   client_86ebc2f   766  0.178    0.529
10  client_8ddc46d   826  0.450    0.528
11  client_a80fca3   214  0.173    0.589
12  client_e547b89  1061  0.256    0.524
13  client_e5c2aa2   685  0.709    0.590
14  client_fef1a8f   206  0.583    0.596


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.